In [ ]:
TARGET_WORDS = ['yes', 'no', 'on', 'off', 'stop']
LABELS = ['silence', 'unknown'] + TARGET_WORDS

N_MFCC = 40
N_FRAMES = 101
N_INPUT = N_MFCC * N_FRAMES
N_OUTPUT = 7
FIXED_SCALE_IN = 1024.0
FIXED_SCALE_OUT = 4096.0

N_FFT = 400
HOP_LENGTH = 160
N_MELS = 40

INPUT_RATE = 48000
SAMPLE_RATE = 16000

STEP_SECONDS = 0.1
STEP_SAMPLES_RAW = int(INPUT_RATE * STEP_SECONDS)        
STEP_SAMPLES = int(SAMPLE_RATE * STEP_SECONDS)

WINDOW_SAMPLES = 16000         

running = False
COOLDOWN_PERIOD = 2.0
COMMAND_LOCKOUT = 1.5
last_command_time = 0
last_recognized_word = None

BLINK_INTERVAL = 0.02
RED_POS   = 0
BLUE_POS  = 2
is_blinking = False
color_index = 0
color_shifts = [RED_POS, BLUE_POS]
last_blink_time = 0

UNKNOWN_LOGIT_OFFSET = 1.4  
HISTORY_LEN = 3    
prediction_history = []






import sys
import numpy as np
import time

from pynq import Overlay, allocate
from scipy.signal import stft, resample_poly
from scipy.fftpack import dct

sys.path.append('/home/xilinx/jupyter_notebooks')

overlay = Overlay("final.bit")

pAudio = overlay.audio_codec_ctrl_0
dma = overlay.axi_dma_0
my_ip = overlay.myproject_0
leds = overlay.leds_gpio
rgb_gpio = overlay.rgbleds_gpio
btns = overlay.btns_gpio

pAudio.configure(sample_rate=48000, iic_index=1, uio_name="audio-codec-ctrl")
pAudio.select_microphone()

input_buffer = allocate(shape=(N_INPUT,), dtype=np.int32)
output_buffer = allocate(shape=(8,), dtype=np.int32)

input_buffer[:] = 0
output_buffer[:] = 0

audio_buffer = np.zeros(16000, dtype=np.float32)









def off_state():
    try:
        dma.sendchannel._mmio.write(0x00, 0x00000004)
        dma.recvchannel._mmio.write(0x30, 0x00000004)
    except Exception:
        pass
    
    overlay.leds_gpio.write(0x00, 0)
    rgb_gpio.write(0x00, 0)

def turn_off(): 
    global running, is_blinking, last_recognized_word, last_command_time
    
    leds.write(0x00, 0x0F)
    rgb_gpio.write(0x00, 1 << 1)
    time.sleep(1.0)
    
    running = False
    is_blinking = False
    last_recognized_word = None
    last_command_time = 0
    off_state()
    
    
    
    
    
    

def get_samples(pAudio):
    pAudio.record(STEP_SECONDS)
    
    raw_buffer = np.asarray(pAudio.buffer, dtype=np.int32)
    
    if len(raw_buffer) < STEP_SAMPLES_RAW:
        return np.zeros(STEP_SAMPLES, dtype=np.float32)
        
    samples = raw_buffer[-STEP_SAMPLES_RAW:].copy()
    
    samples = samples.astype(np.int64)
    samples = samples & 0xFFFFFF
    samples[samples & 0x800000 != 0] -= 0x1000000
    
    samples_float = samples.astype(np.float32) / (2**23)
    samples_float = np.clip(samples_float, -1.0, 1.0)
    samples_float -= np.mean(samples_float)
    
    samples_16k = resample_poly(samples_float, 1, 3).astype(np.float32)
    
    if len(samples_16k) > STEP_SAMPLES:
        samples_16k = samples_16k[:STEP_SAMPLES]
    elif len(samples_16k) < STEP_SAMPLES:
        samples_16k = np.pad(samples_16k, (0, STEP_SAMPLES - len(samples_16k)), 'constant')
        
    return samples_16k


def create_mel_weight_matrix(num_mel_filter, num_spectrogram_coeffs, sample_rate, low_freq, high_freq):
    def hz_to_mel(hz):
        return 2595.0 * np.log10(1.0 + hz / 700.0)

    def mel_to_hz(mel):
        return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)
    
    low_mel = hz_to_mel(low_freq)
    higher_mel = hz_to_mel(high_freq)
    
    mel_frequencies = np.linspace(low_mel, higher_mel, num_mel_filter + 2)
    frequency_hz = mel_to_hz(mel_frequencies)

    frequency_ratio = (num_spectrogram_coeffs * 2) * frequency_hz / sample_rate
    frequency_idx = np.floor(frequency_ratio).astype(int)
    
    mel_filter_weights = np.zeros((num_spectrogram_coeffs, num_mel_filter))

    for i in range(num_mel_filter):
        start = frequency_idx[i]
        center = frequency_idx[i + 1]
        end = frequency_idx[i + 2]

        for j in range(start, center):
            if center != start:
                mel_filter_weights[j, i] = (j - start) / (center - start)

        for j in range(center, end):
            if end != center:
                mel_filter_weights[j, i] = (end - j) / (end - center)

    return mel_filter_weights

MEL_FILTER_WEIGHTS = create_mel_weight_matrix(
    num_mel_filter=N_MELS,
    num_spectrogram_coeffs=(N_FFT // 2 + 1),
    sample_rate=SAMPLE_RATE,
    low_freq=20.0,
    high_freq=4000.0
)

HANN_WINDOW = np.hanning(N_FFT)
HANN_SUM = np.sum(HANN_WINDOW)

def extract_mfcc(signal, sample_rate=SAMPLE_RATE, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS):
    signal = np.asarray(signal, dtype=np.float32)
    padding = n_fft // 2
    padded_signal = np.pad(signal, (padding, padding), mode='reflect')

    f, t, Zxx = stft(
        padded_signal,
        fs=sample_rate,
        window='hann',
        nperseg=n_fft,
        noverlap=n_fft - hop_length,
        boundary=None,
        padded=False
    )
    
    amplitudes = np.abs(Zxx)
    spectrogram = amplitudes * HANN_SUM
    spectrogram = spectrogram.T
    
    mel_spectrogram = np.dot(spectrogram, MEL_FILTER_WEIGHTS)
    log_mel_spectrogram = np.log(mel_spectrogram + 1e-6)

    mfcc = dct(log_mel_spectrogram, type=2, axis=-1, norm='ortho')
    mfcc = mfcc[..., :n_mfcc]
  
    mfcc = mfcc.T
    mfcc = np.expand_dims(mfcc, axis=0)
    mfcc = np.expand_dims(mfcc, axis=-1)

    return mfcc.astype(np.float32)

def get_MFCC(chunk):
    if len(chunk) != STEP_SAMPLES:
        return False
        
    audio_buffer[:-STEP_SAMPLES] = audio_buffer[STEP_SAMPLES:]
    audio_buffer[-STEP_SAMPLES:] = chunk
    
    features = extract_mfcc(audio_buffer)
    
    mfcc_int32 = np.clip(np.round(features * FIXED_SCALE_IN), -2147483648, 2147483647).astype(np.int32)
    
    input_buffer[:] = mfcc_int32.flatten(order='C')
    output_buffer[:] = 0
    
    return True

In [ ]:
import numpy as np
import time

TEST_SAMPLES_COUNT = 50

stats_faza1 = [] 
stats_faza2  = [] 
stats_faza3  = [] 
stats_faza4  = [] 
stats_total  = [] 

print("Pritisnite BTN0 za pokretanje")
while not running:
    if btns.read() & 0x01:
        running = True
        time.sleep(0.5)
        leds.write(0x00, 0x0F)
        rgb_gpio.write(0x00, 1 << 0)
        time.sleep(0.5)

        leds.write(0x00, 0x00)
        rgb_gpio.write(0x00, 0x00)

    time.sleep(0.1)

print("Sustav aktiviran")

while running:
    if btns.read() & 0x01:
        turn_off()
        break

    t0 = time.perf_counter()
    chunk = get_samples(pAudio)
    t1 = time.perf_counter()

    t_mfcc_start = time.perf_counter()
    has_mfcc = get_MFCC(chunk)
    t_mfcc_end = time.perf_counter()

    if not has_mfcc:
        continue

    t2 = time.perf_counter()
    my_ip.write(0x00, 1)
    dma.recvchannel.transfer(output_buffer)
    dma.sendchannel.transfer(input_buffer)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    t3 = time.perf_counter()

    output_data = np.array(output_buffer, dtype=np.int32)
    model_output = np.zeros(7, dtype=np.float32)

    for i in range(7):
        bit_offset = i * 24
        word_idx = bit_offset // 32
        shift = bit_offset % 32
        val = (output_data[word_idx] >> shift) & 0xFFFFFF

        if shift > 8:
            bits1 = 32 - shift
            bits2 = 24 - bits1
            mask1 = (1 << bits1) - 1
            mask2 = (1 << bits2) - 1
            part1 = (output_data[word_idx] >> shift) & mask1
            part2 = output_data[word_idx + 1] & mask2
            val = part1 | (part2 << bits1)

        if val & 0x800000:
            val -= 0x1000000

        model_output[i] = val

    logits = model_output.astype(np.float32) / FIXED_SCALE_OUT
    logits[1] -= UNKNOWN_LOGIT_OFFSET
    logits_shifted = logits - np.max(logits)

    probabilities = np.exp(logits_shifted)
    probabilities /= np.sum(probabilities)

    prediction_history.append(probabilities)
    if len(prediction_history) > HISTORY_LEN:
        prediction_history.pop(0)

    predicted_index = int(np.argmax(probabilities))
    confidence = float(probabilities[predicted_index])
    predicted_label = LABELS[predicted_index]

    current_time = time.time()

    if predicted_label in ['silence', 'unknown']:
        if (current_time - last_command_time) > 0.4:
            last_recognized_word = None

    if predicted_label in TARGET_WORDS:
        if (predicted_label != last_recognized_word) or ((current_time - last_command_time) >= COMMAND_LOCKOUT):

            if (current_time - last_command_time) >= COMMAND_LOCKOUT:
                if predicted_label == 'on':
                    leds.write(0x00, 15)
                elif predicted_label == 'off':
                    leds.write(0x00, 0)
                elif predicted_label == 'yes':
                    is_blinking = True
                elif predicted_label == 'no':
                    is_blinking = False
                    rgb_gpio.write(0x00, 0)
                elif predicted_label == 'stop':
                    is_blinking = False
                    leds.write(0x00, 0)
                    rgb_gpio.write(0x00, 0)

                t4 = time.perf_counter()

                l_faza1 = (t1 - t0) * 1000.0
                l_faza2 = (t_mfcc_end - t_mfcc_start) * 1000.0
                l_faza3 = (t3 - t2) * 1000.0
                l_faza4 = (t4 - t3) * 1000.0
                l_tot   = (t4 - t0) * 1000.0

                stats_faza1.append(l_faza1)
                stats_faza2.append(l_faza2)
                stats_faza3.append(l_faza3)
                stats_faza4.append(l_faza4)
                stats_total.append(l_tot)

                sample_num = len(stats_total)
                print(f"[{sample_num:02d}/{TEST_SAMPLES_COUNT}] Detektirano: '{predicted_label.upper():<4}' | Ukupno: {l_tot:.2f} ms (FPGA: {l_faza3:.2f} ms)")

                if sample_num >= TEST_SAMPLES_COUNT:
                    print(f"1. Faza Prosjek: {np.mean(stats_faza1):.2f} ms")
                    print(f"1. Faza Min: {np.min(stats_faza1):.2f} ms")
                    print(f"1. Faza Max: {np.max(stats_faza1):.2f} ms")
                    print(f"1. Faza Std Dev: {np.std(stats_faza1):.2f}")

                    print(f"2. Faza Prosjek: {np.mean(stats_faza2):.2f} ms")
                    print(f"2. Faza Min: {np.min(stats_faza2):.2f} ms")
                    print(f"2. Faza Max: {np.max(stats_faza2):.2f} ms")
                    print(f"2. Faza Std Dev: {np.std(stats_faza2):.2f}")

                    print(f"3. Faza Prosjek: {np.mean(stats_faza3):.2f} ms")
                    print(f"3. Faza Min: {np.min(stats_faza3):.2f} ms")
                    print(f"3. Faza Max: {np.max(stats_faza3):.2f} ms")
                    print(f"3. Faza Std Dev: {np.std(stats_faza3):.2f}")

                    print(f"4. Faza Prosjek: {np.mean(stats_faza4):.2f} ms")
                    print(f"4. Faza Min: {np.min(stats_faza4):.2f} ms")
                    print(f"4. Faza Max: {np.max(stats_faza4):.2f} ms")
                    print(f"4. Faza Std Dev: {np.std(stats_faza4):.2f}")

                    print(f"Ukupno vrijeme odziva Prosjek: {np.mean(stats_total):.2f} ms")
                    print(f"Ukupno vrijeme odziva Std Dev: {np.std(stats_total):.2f}")
                    turn_off()
                    break

                last_command_time = current_time

            last_recognized_word = predicted_label